In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [52]:
def make_toy_data(n=5000, seed=42):
    rng = np.random.default_rng(seed)

    df = pd.DataFrame({
        "age": rng.integers(18, 75, size=n),
        "income": rng.normal(80_000, 25_000, size=n),
        "sessions_7d": rng.poisson(5, size=n),
        "country": rng.choice(["US", "CA", "UK", "DE", None], size=n, p=[0.55, 0.15, 0.12, 0.10, 0.08]),
        "device": rng.choice(["ios", "android", "web", None], size=n, p=[0.35, 0.35, 0.22, 0.08]),
        "ad_type": rng.choice(["video", "image", "carousel"], size=n),
    })

    # Add some missing numeric values
    df.loc[rng.choice(n, size=int(0.05 * n), replace=False), "income"] = np.nan
    df.loc[rng.choice(n, size=int(0.03 * n), replace=False), "sessions_7d"] = np.nan

    # Create a synthetic binary target with signal
    logits = (
        -3.0
        + 0.025 * (df["age"].fillna(40) - 40)
        + 0.000025 * (df["income"].fillna(80_000) - 80_000)
        + 0.25 * df["sessions_7d"].fillna(5)
        + 0.7 * (df["country"] == "US").astype(float)
        + 0.5 * (df["device"] == "ios").astype(float)
        + 0.6 * (df["ad_type"] == "video").astype(float)
    )

    probs = 1 / (1 + np.exp(-logits))
    df["label"] = rng.binomial(1, probs).astype(np.float32)

    return df

df = make_toy_data(n=5000)
print(df.head())

   age         income  sessions_7d country   device ad_type  label
0   23   70597.395236          NaN      DE  android   video    0.0
1   62   68547.954958          4.0      US  android   image    0.0
2   55   99224.146789          5.0      CA      web   video    0.0
3   43  100477.604223          6.0      CA  android   video    0.0
4   42   66622.821166          6.0      UK      ios   video    0.0


### Create Train Test Split

In [53]:
random_state = 42
test_size = 0.2
target_col = "label"

train_df, val_df = train_test_split(
    df,
    test_size=test_size,
    random_state=random_state,
    stratify=df[target_col],  # useful for binary classification
)
print(train_df.shape, (train_df[target_col] == 1).mean())
print(val_df.shape, (val_df[target_col] == 1).mean() )

(4000, 7) 0.3275
(1000, 7) 0.328


### Preprocess

In [54]:
def build_preprocessor(df: pd.DataFrame, target_col: str):
    feature_cols = [c for c in df.columns if c != target_col]

    categorical_cols = [
        c for c in feature_cols
        if df[c].dtype == "object" or str(df[c].dtype).startswith("category")
    ]

    numeric_cols = [
        c for c in feature_cols
        if c not in categorical_cols
    ]

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols),
        ]
    )

    return preprocessor, feature_cols
    
def preprocess_data(train_df, val_df, target_col):
    preprocessor, feature_cols = build_preprocessor(train_df, target_col)

    X_train = preprocessor.fit_transform(train_df[feature_cols])
    X_val = preprocessor.transform(val_df[feature_cols])

    y_train = train_df[target_col].values.astype(np.float32)
    y_val = val_df[target_col].values.astype(np.float32)

    X_train = X_train.astype(np.float32)
    X_val = X_val.astype(np.float32)

    return X_train, y_train, X_val, y_val, preprocessor
    
X_train, y_train, X_val, y_val, preprocessor = preprocess_data(
    train_df,
    val_df,
    target_col,
)

### Create Dataset

In [55]:
class PandasTorchDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [56]:
train_dataset = PandasTorchDataset(X_train, y_train)
val_dataset = PandasTorchDataset(X_val, y_val)

### Model

In [57]:
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.1):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # raw logits, no sigmoid here

### Train Loop

In [58]:
# -----------------------------
# 6. Train / eval loops
# -----------------------------

def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples

### Eval Loop

In [59]:
@torch.no_grad()
def evaluate(model, dataloader, loss_fn, device):
    model.eval()

    total_loss = 0.0
    total_examples = 0

    all_probs = []
    all_targets = []

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        probs = torch.sigmoid(logits)

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        all_probs.append(probs.cpu())
        all_targets.append(y_batch.cpu())

    avg_loss = total_loss / total_examples

    all_probs = torch.cat(all_probs).numpy().ravel()
    all_targets = torch.cat(all_targets).numpy().ravel()

    preds = (all_probs >= 0.5).astype(int)
    accuracy = (preds == all_targets).mean()

    return avg_loss, accuracy, all_probs, all_targets

### Full Training Fn

In [60]:
# -----------------------------
# 7. Full training function
# -----------------------------

def train_model(
    train_dataset: Dataset,
    val_dataset: Dataset,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    loss_fn: "Loss",
    batch_size: int,
):

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    input_dim = X_train.shape[1]

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_fn,
            device,
        )

        val_loss, val_acc, val_probs, val_targets = evaluate(
            model,
            val_loader,
            loss_fn,
            device,
        )

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f}"
        )

    return model, preprocessor

In [64]:
weight_decay = 1e-4
lr = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 512
epochs = 10

model = BinaryClassifier(
    input_dim=X_train.shape[1],
    hidden_dims=(128, 64),
    dropout=0.1,
).to(device)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay,
)

trained_model, preprocessor = train_model(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    batch_size=batch_size
)

Epoch 001 | train_loss=0.6574 | val_loss=0.6351 | val_acc=0.6760
Epoch 002 | train_loss=0.6213 | val_loss=0.6021 | val_acc=0.6900
Epoch 003 | train_loss=0.5921 | val_loss=0.5744 | val_acc=0.6980
Epoch 004 | train_loss=0.5679 | val_loss=0.5549 | val_acc=0.7170
Epoch 005 | train_loss=0.5545 | val_loss=0.5455 | val_acc=0.7250
Epoch 006 | train_loss=0.5508 | val_loss=0.5440 | val_acc=0.7290
Epoch 007 | train_loss=0.5486 | val_loss=0.5424 | val_acc=0.7270
Epoch 008 | train_loss=0.5460 | val_loss=0.5418 | val_acc=0.7300
Epoch 009 | train_loss=0.5449 | val_loss=0.5413 | val_acc=0.7260
Epoch 010 | train_loss=0.5452 | val_loss=0.5407 | val_acc=0.7280


### Inference

In [66]:
@torch.no_grad()
def predict_proba(model, preprocessor, new_df: pd.DataFrame):
    model.eval()

    X = preprocessor.transform(new_df)
    X = X.astype(np.float32)

    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

    logits = model(X_tensor)
    probs = torch.sigmoid(logits)

    return probs.cpu().numpy().ravel()

probs = predict_proba(model, preprocessor, val_df)
preds = (probs >= 0.5).astype(int)
preds

array([0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0,
       0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0,